# Monitoring & Drift Detection

Models degrade in production. This notebook covers how to detect and respond to drift.

1. **Types of Drift** - Data drift, concept drift, prediction drift
2. **Statistical Tests** - PSI, KS test, chi-squared test
3. **Evidently AI** - Automated drift monitoring dashboards
4. **Alerting Strategy** - When to retrain

**Scenario**: Simulate drift on Breast Cancer dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from scipy import stats

sns.set_theme(style="whitegrid")

## Types of Drift

| Type | What Changes | Example |
|------|-------------|--------|
| **Data drift** | Input feature distributions P(X) | Customer demographics shift |
| **Concept drift** | Relationship between X and y: P(y\|X) | Fraud patterns evolve |
| **Prediction drift** | Model output distribution P(ŷ) | Predictions become more extreme |
| **Label drift** | True label distribution P(y) | Class imbalance changes |

In [ ]:
# Train a model on "reference" (training) data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", GradientBoostingClassifier(n_estimators=100, random_state=42)),
])
pipe.fit(X_train, y_train)

baseline_acc = accuracy_score(y_test, pipe.predict(X_test))
print(f"Baseline accuracy: {baseline_acc:.4f}")

In [ ]:
# Simulate data drift: shift features in "production" data
np.random.seed(42)

def simulate_drift(X, drift_magnitude=1.0):
    """Add systematic shift to simulate feature drift."""
    X_drifted = X.copy()
    # Shift a few important features
    for col in X.columns[:5]:
        X_drifted[col] = X_drifted[col] + drift_magnitude * X_drifted[col].std()
    return X_drifted

# No drift, mild drift, severe drift
scenarios = {
    "No drift": X_test.copy(),
    "Mild drift (0.5σ)": simulate_drift(X_test, 0.5),
    "Moderate drift (1σ)": simulate_drift(X_test, 1.0),
    "Severe drift (2σ)": simulate_drift(X_test, 2.0),
}

for name, X_drifted in scenarios.items():
    acc = accuracy_score(y_test, pipe.predict(X_drifted))
    print(f"{name:25s}: accuracy = {acc:.4f} (delta = {acc - baseline_acc:+.4f})")

## Statistical Drift Detection

In [ ]:
def detect_drift_ks(reference, current, threshold=0.05):
    """Detect drift using Kolmogorov-Smirnov test per feature."""
    results = []
    for col in reference.columns:
        stat, pvalue = stats.ks_2samp(reference[col], current[col])
        results.append({
            "feature": col,
            "ks_statistic": stat,
            "p_value": pvalue,
            "drift_detected": pvalue < threshold,
        })
    return pd.DataFrame(results)

# Test on severe drift scenario
X_severe = scenarios["Severe drift (2σ)"]
drift_results = detect_drift_ks(X_train, X_severe)

drifted_features = drift_results[drift_results["drift_detected"]]
print(f"Drift detected in {len(drifted_features)}/{len(drift_results)} features\n")
print(drifted_features.sort_values("ks_statistic", ascending=False).head(10).to_string(index=False))

In [ ]:
# Visualize drift in top features
top_drifted = drifted_features.nlargest(4, "ks_statistic")["feature"].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, feat in zip(axes.flat, top_drifted):
    ax.hist(X_train[feat], bins=30, alpha=0.6, label="Reference", color="teal", density=True)
    ax.hist(X_severe[feat], bins=30, alpha=0.6, label="Production", color="coral", density=True)
    ks = drift_results[drift_results["feature"] == feat]["ks_statistic"].values[0]
    ax.set_title(f"{feat} (KS={ks:.3f})")
    ax.legend()

plt.suptitle("Feature Distribution Drift", fontsize=14)
plt.tight_layout()
plt.show()

## Population Stability Index (PSI)

PSI quantifies how much a distribution has shifted:

$$PSI = \sum_{i=1}^{n} (p_i^{\text{actual}} - p_i^{\text{expected}}) \times \ln\frac{p_i^{\text{actual}}}{p_i^{\text{expected}}}$$

| PSI | Interpretation |
|-----|---------------|
| < 0.1 | No significant shift |
| 0.1 - 0.2 | Moderate shift, investigate |
| > 0.2 | Significant shift, action required |

In [ ]:
def calculate_psi(reference, current, bins=10):
    """Calculate Population Stability Index."""
    # Create bins from reference distribution
    breakpoints = np.linspace(np.min(reference), np.max(reference), bins + 1)
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    
    ref_counts = np.histogram(reference, bins=breakpoints)[0] / len(reference)
    cur_counts = np.histogram(current, bins=breakpoints)[0] / len(current)
    
    # Avoid division by zero
    ref_counts = np.clip(ref_counts, 1e-4, None)
    cur_counts = np.clip(cur_counts, 1e-4, None)
    
    psi = np.sum((cur_counts - ref_counts) * np.log(cur_counts / ref_counts))
    return psi

# Calculate PSI for each feature under different drift levels
print(f"{'Feature':<30} {'No drift':>10} {'Mild':>10} {'Moderate':>10} {'Severe':>10}")
print("-" * 72)
for feat in X.columns[:10]:
    psi_values = []
    for name, X_scenario in scenarios.items():
        psi = calculate_psi(X_train[feat].values, X_scenario[feat].values)
        psi_values.append(f"{psi:.4f}")
    print(f"{feat:<30} {'  '.join(psi_values)}")

## Evidently AI (Optional)

Evidently generates interactive drift reports with a single function call.

```python
# pip install evidently
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, TargetDriftPreset

report = Report(metrics=[DataDriftPreset()])
report.run(reference_data=X_train, current_data=X_severe)
report.save_html("drift_report.html")
# Open drift_report.html in a browser for interactive dashboard
```

## Monitoring Strategy

```
Production Data ──> Drift Detection ──> Alert? ──> Retrain Pipeline
                         │
                    ┌────┴────┐
                    │ Compare │
                    └────┬────┘
                         │
              Reference (Training) Data
```

| What to Monitor | How | Alert Threshold |
|----------------|-----|-----------------|
| Feature distributions | PSI, KS test | PSI > 0.2 |
| Prediction distribution | PSI on P(ŷ) | PSI > 0.1 |
| Model performance | Accuracy on labeled data | Drop > 2% from baseline |
| Latency | p50, p95, p99 | p95 > 200ms |
| Error rate | 4xx/5xx responses | > 1% |

## Key Takeaways

1. **All models degrade** - the question is when, not if
2. **Monitor inputs AND outputs** - data drift doesn't always cause performance drop (and vice versa)
3. **PSI is the industry standard** for drift quantification
4. **Automate retraining triggers** - but always validate before deploying
5. **Ground truth delay** is the biggest challenge - you may not get labels for weeks/months